# Exploratory Data Analysis (EDA)

In this notebook, we explore historical sales and inventory data to understand
how demand behaves across SKUs, categories, and time.

The goal is not to create as many plots as possible, but to extract insights
that will guide forecasting and inventory decisions later.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
# Load processed data
from src.utils import load_processed_data

sales_df, inventory_df, products_df, suppliers_df = load_processed_data()

In [ ]:
# Convert date columns to datetime
sales_df["date"] = pd.to_datetime(sales_df["date"])
inventory_df["snapshot_date"] = pd.to_datetime(inventory_df["snapshot_date"])

## Dataset Preparation for Analysis

We combine sales data with product attributes to enable category-level
and SKU-level analysis.

In [ ]:
# Merge sales data with product details
sales_enriched = sales_df.merge(
    products_df,
    on="sku_id",
    how="left"
)

sales_enriched.head()

## Overall Demand Trend

We start by understanding how total demand behaves over time.
This helps identify seasonality, growth trends, and anomalies.

In [ ]:
daily_demand = sales_enriched.groupby("date")["units_sold"].sum()

plt.figure(figsize=(12, 4))
daily_demand.plot()
plt.title("Total Units Sold Over Time")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.show()

## Category-Level Demand Patterns

Different categories behave very differently in terms of volume,
seasonality, and volatility.

In [ ]:
category_demand = (
    sales_enriched
    .groupby(["date", "category"])["units_sold"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=category_demand, x="date", y="units_sold", hue="category")
plt.title("Category-wise Demand Over Time")
plt.show()

## SKU Velocity Analysis

We identify fast-moving and slow-moving SKUs based on average weekly demand.

In [ ]:
weekly_sales = (
    sales_enriched
    .set_index("date")
    .groupby("sku_id")["units_sold"]
    .resample("W")
    .sum()
    .reset_index()
)

sku_velocity = weekly_sales.groupby("sku_id")["units_sold"].mean()

plt.figure(figsize=(6, 4))
sku_velocity.hist(bins=50)
plt.title("Distribution of Average Weekly SKU Demand")
plt.xlabel("Average Weekly Units Sold")
plt.show()

## Revenue Concentration (Pareto Insight)

A small percentage of SKUs usually contributes to the majority of sales.
This insight motivates ABC classification later.

In [ ]:
sales_enriched["revenue"] = (
    sales_enriched["units_sold"] * sales_enriched["selling_price"]
)

sku_revenue = (
    sales_enriched
    .groupby("sku_id")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

cumulative_revenue = sku_revenue.cumsum() / sku_revenue.sum()

plt.figure(figsize=(6, 4))
cumulative_revenue.reset_index(drop=True).plot()
plt.axhline(0.8, color="red", linestyle="--")
plt.title("Cumulative Revenue Contribution by SKU")
plt.xlabel("SKU Rank")
plt.ylabel("Cumulative Revenue Share")
plt.show()

## Demand Volatility Analysis

High demand volatility increases forecast uncertainty and
requires higher safety stock.

In [ ]:
sku_volatility = (
    weekly_sales
    .groupby("sku_id")["units_sold"]
    .agg(["mean", "std"])
)

sku_volatility["cv"] = sku_volatility["std"] / sku_volatility["mean"]

plt.figure(figsize=(6, 4))
sku_volatility["cv"].hist(bins=50)
plt.title("Demand Volatility (Coefficient of Variation)")
plt.xlabel("CV")
plt.show()

## Stockout Signal Exploration

We approximate stockout situations by identifying periods where
inventory levels hit zero.

In [ ]:
stockout_events = inventory_df[inventory_df["on_hand_qty"] == 0]

stockout_counts = (
    stockout_events
    .groupby("sku_id")
    .size()
    .sort_values(ascending=False)
)

stockout_counts.head()

## Supplier Lead Time Variability

Lead time uncertainty directly affects reorder point and safety stock.

In [ ]:
plt.figure(figsize=(6, 4))
suppliers_df["lead_time_days"].hist(bins=30)
plt.title("Distribution of Supplier Lead Times")
plt.xlabel("Lead Time (Days)")
plt.show()

## Key Takeaways from EDA

- Demand is highly skewed across SKUs
- Different categories exhibit different seasonality and volatility
- A small number of SKUs drive most revenue
- Many SKUs show high demand variability
- Supplier lead times vary widely and add replenishment risk

These insights justify:
- SKU segmentation (ABC/XYZ)
- Segment-specific forecasting models
- Variable safety stock policies